# 1. Boosting – Insurance Charges Prediction
**Dataset:** `insurance_pre.csv` (1338 rows) | **Task:** Regression – predict `charges`  
**Algorithms (4):** AdaBoost, Gradient Boosting, Histogram Gradient Boosting, XGBoost

In [ ]:
import subprocess, sys, warnings
warnings.filterwarnings('ignore')
try:
    import xgboost
except ImportError:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'xgboost'])
    import xgboost
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.ensemble import AdaBoostRegressor, GradientBoostingRegressor, HistGradientBoostingRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from xgboost import XGBRegressor
sns.set_style('whitegrid')

## 1. Load & explore the data

In [ ]:
df = pd.read_csv('data/insurance_pre.csv')
print(df.shape)
display(df.head())
print(df.info())
print('Missing values:\n', df.isnull().sum())
print('Duplicates:', df.duplicated().sum())
display(df.describe())

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(16, 4))
sns.histplot(df['charges'], kde=True, ax=ax[0]); ax[0].set_title('Distribution of charges')
sns.boxplot(x='smoker', y='charges', data=df, ax=ax[1]); ax[1].set_title('Charges vs smoker')
sns.scatterplot(x='bmi', y='charges', hue='smoker', data=df, ax=ax[2]); ax[2].set_title('BMI vs charges')
plt.tight_layout(); plt.show()

## 2. Preprocessing
Categorical columns (`sex`, `smoker`) are one-hot encoded; tree-based boosting needs no scaling.

In [ ]:
df = df.drop_duplicates()
X = pd.get_dummies(df.drop('charges', axis=1), drop_first=True)
y = df['charges']
plt.figure(figsize=(6, 5))
sns.heatmap(pd.concat([X, y], axis=1).corr(), annot=True, fmt='.2f', cmap='coolwarm'); plt.title('Correlation heatmap'); plt.show()
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(X_train.shape, X_test.shape)

## 3. Train the 4 boosting algorithms

In [ ]:
models = {
    'AdaBoost': AdaBoostRegressor(estimator=DecisionTreeRegressor(max_depth=4), n_estimators=100, learning_rate=0.5, random_state=42),
    'Gradient Boosting': GradientBoostingRegressor(n_estimators=100, learning_rate=0.1, max_depth=3, random_state=42),
    'Hist Gradient Boosting': HistGradientBoostingRegressor(max_iter=100, learning_rate=0.1, max_depth=3, random_state=42),
    'XGBoost': XGBRegressor(n_estimators=100, learning_rate=0.1, max_depth=3, random_state=42),
}
rows, preds = [], {}
for name, m in models.items():
    m.fit(X_train, y_train)
    p = m.predict(X_test); preds[name] = p
    cv = cross_val_score(m, X, y, cv=5, scoring='r2')
    rows.append({'Model': name,
                 'Train R2': r2_score(y_train, m.predict(X_train)),
                 'Test R2': r2_score(y_test, p),
                 'MAE': mean_absolute_error(y_test, p),
                 'RMSE': np.sqrt(mean_squared_error(y_test, p)),
                 'CV R2 (5-fold)': cv.mean()})
results = pd.DataFrame(rows).set_index('Model').round(4).sort_values('Test R2', ascending=False)
display(results)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(14, 4))
results['Test R2'].plot(kind='bar', ax=ax[0], color='steelblue'); ax[0].set_title('Test R² by model'); ax[0].set_ylim(0.7, 1)
results['RMSE'].plot(kind='bar', ax=ax[1], color='indianred'); ax[1].set_title('RMSE by model (lower is better)')
plt.tight_layout(); plt.show()

best_name = results.index[0]
plt.figure(figsize=(5, 5))
plt.scatter(y_test, preds[best_name], alpha=0.6)
plt.plot([y.min(), y.max()], [y.min(), y.max()], 'r--')
plt.xlabel('Actual charges'); plt.ylabel('Predicted charges'); plt.title(f'Actual vs Predicted – {best_name}'); plt.show()

## 4. Hyper-parameter tuning of the best sklearn model (Gradient Boosting)

In [ ]:
grid = GridSearchCV(GradientBoostingRegressor(random_state=42),
                    {'n_estimators': [50, 100, 200], 'learning_rate': [0.03, 0.05, 0.1], 'max_depth': [2, 3, 4]},
                    cv=5, scoring='r2', n_jobs=-1)
grid.fit(X_train, y_train)
tuned_pred = grid.predict(X_test)
print('Best params:', grid.best_params_)
print('Tuned Gradient Boosting  Test R2 = %.4f | MAE = %.2f | RMSE = %.2f' % (
    r2_score(y_test, tuned_pred), mean_absolute_error(y_test, tuned_pred), np.sqrt(mean_squared_error(y_test, tuned_pred))))

In [ ]:
imp = pd.Series(grid.best_estimator_.feature_importances_, index=X.columns).sort_values()
imp.plot(kind='barh', figsize=(6, 3), title='Feature importance (tuned Gradient Boosting)'); plt.show()

## 5. Conclusion
- All four boosting algorithms were trained on the same 80/20 split and compared on R², MAE, RMSE and 5-fold CV R².
- **AdaBoost** is clearly the weakest (higher MAE/RMSE); the gradient-based methods (Gradient Boosting, HistGB, XGBoost) fit noticeably better. Write the final ranking from the results table above.
- `smoker` is by far the most important predictor, followed by `bmi` and `age`.
- Hyper-parameter tuning (GridSearchCV) picks a simpler, more regularised model; its test R² is about the same as the default one, i.e. the defaults were already near the optimum for this data.